# Avaliação do SAM na validação congelada

O experimento aplica as três modalidades verificadas na integração técnica às **294 imagens da validação**. Esta etapa seleciona o protocolo SAM que será congelado antes do teste, mas ainda não mede o efeito final.

Fluxo: **DR-IFT congelada → prompts automáticos → SAM congelado → comparação pareada por imagem**. Teste e FUSeg continuam fechados.


## O que já está decidido

- DR-IFT seed 42, época 56 e threshold 0,18;
- SAM 1 ViT-B sem fine-tuning;
- modos `box`, `mask_input` e `box+mask`;
- prompts produzidos somente pela DR-IFT;
- Dice macro por imagem como métrica primária;
- IC 95% bootstrap pareado e Holm nos três contrastes de Dice;
- escolha pela maior média de Dice, com tolerância de 0,0001 e desempate conservador: box, mask e box+mask.

A regra foi registrada antes desta execução para que o resultado não possa mudar o critério de escolha.


## 1. Preparar o Colab

O Dataset será copiado para o disco temporário apenas para acelerar a leitura. O projeto e as saídas permanecem no Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import json, os, shutil, subprocess, sys, urllib.request, yaml

PROJECT_ROOT=Path('/content/drive/MyDrive/Projeto_TCC_Lucas')
DATA_ROOT=Path('/content/tcc_data')
%cd {PROJECT_ROOT}
assert PROJECT_ROOT.is_dir(), f'Projeto ausente: {PROJECT_ROOT}'
source=PROJECT_ROOT/'Dataset'
assert source.is_dir(), f'Dataset ausente: {source}'
shutil.copytree(source,DATA_ROOT/'Dataset',dirs_exist_ok=True)
assert (DATA_ROOT/'Dataset/DFUC2022_train_images/100001.jpg').is_file()
gpu=subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],capture_output=True,text=True).stdout.strip()
print('Cache local pronto | GPU:',gpu)


## 2. Instalar a versão congelada do SAM

O código oficial permanece fixado no mesmo commit usado no smoke test. O checkpoint já existente no Drive será reutilizado.


In [ ]:
SAM_COMMIT = "6fdee8f2727f4506cfbbe553e23b895e27956588"
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"git+https://github.com/facebookresearch/segment-anything.git@{SAM_COMMIT}",
    ],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "pyyaml", "pytest", "matplotlib"], check=True
)
sam_dir = PROJECT_ROOT / "models/sam"
sam_dir.mkdir(parents=True, exist_ok=True)
sam_checkpoint = sam_dir / "sam_vit_b_01ec64.pth"
sam_url = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"
if not sam_checkpoint.is_file():
    print("Baixando checkpoint oficial ViT-B...")
    urllib.request.urlretrieve(sam_url, sam_checkpoint)
print("SAM:", sam_checkpoint, f"{sam_checkpoint.stat().st_size/1024**2:.1f} MiB")


## 3. Conferir o protocolo antes da inferência

Esta célula verifica os arquivos, os hashes dos manifests, as 294 imagens e a identidade do checkpoint. Se falhar, não substitua arquivos nem altere a regra.


In [ ]:
CONFIG = PROJECT_ROOT / "configs/sam_validation.yaml"
cfg = yaml.safe_load(CONFIG.read_text())
network_run = PROJECT_ROOT / cfg["network"]["run_directory"]
network_checkpoint = network_run / cfg["network"]["checkpoint_filename"]
required = [
    CONFIG,
    network_run / "config_used.yaml",
    network_checkpoint,
    sam_checkpoint,
    PROJECT_ROOT / cfg["data"]["manifest"],
    PROJECT_ROOT / cfg["data"]["fixed_visualization_manifest"],
]
for path in required:
    assert path.is_file(), f"Arquivo ausente: {path}"
import csv, hashlib


def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


manifest = PROJECT_ROOT / cfg["data"]["manifest"]
with manifest.open(encoding="utf-8-sig", newline="") as handle:
    rows = list(csv.DictReader(handle))
assert len(rows) == 294 and all(r["split"] == "validation" for r in rows)
assert sha(manifest) == cfg["data"]["manifest_sha256"]
print("Protocolo:", cfg["stage"])
print(
    "DR-IFT:",
    cfg["network"]["run_id"],
    "| época",
    cfg["network"]["checkpoint_epoch"],
    "| threshold",
    cfg["network"]["threshold"],
)
print("Validação congelada:", len(rows), "imagens | hashes aprovados")


## 4. Rodar os testes

Os testes de integração verificam a interface dos prompts. Os testes de validação verificam a seleção, o pareamento, o Boundary IoU e o bloqueio das partições finais.


In [ ]:
env = os.environ.copy()
env["PYTHONPATH"] = str(PROJECT_ROOT / "src") + os.pathsep + env.get("PYTHONPATH", "")
result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "-q",
        "tests/test_sam_refinement.py",
        "tests/test_sam_validation.py",
    ],
    text=True,
    capture_output=True,
    env=env,
)
print(result.stdout)
print(result.stderr)
result.check_returncode()


## 5. Executar as 294 imagens

Cada imagem passa uma vez pela DR-IFT e pelos três modos do SAM. A saída mostra o progresso a cada dez imagens. Não interrompa enquanto não aparecer `status: completed`.


In [ ]:
def run_stream(command):
    process = subprocess.Popen(
        command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    for line in process.stdout:
        print(line, end="")
    code = process.wait()
    print(f"\nreturncode={code}")
    if code:
        raise subprocess.CalledProcessError(code, command)
    return code


run_stream(
    [
        sys.executable,
        "scripts/run_sam_validation.py",
        "--project-root",
        str(PROJECT_ROOT),
        "--data-root",
        str(DATA_ROOT),
        "--config",
        "configs/sam_validation.yaml",
    ]
)


## 6. Ler o resultado quantitativo

A primeira tabela mostra o desempenho absoluto. A segunda mostra quanto cada modo mudou o Dice da mesma imagem em relação à DR-IFT. O intervalo que cruza zero indica incerteza sobre a direção média na validação.


In [ ]:
import pandas as pd
from IPython.display import display

runs = sorted(
    (PROJECT_ROOT / "results/day13/SAM_validation").glob("SAM1_ViTB_DRIFT_seed42_validation_*")
)
assert runs, "Nenhuma execução de validação do SAM encontrada"
RUN = runs[-1]
summary = pd.read_csv(RUN / "variant_summary.csv")
effects = pd.read_csv(RUN / "paired_comparisons.csv")
selection = json.loads((RUN / "selection.json").read_text())
display(
    summary[
        [
            "variant",
            "images",
            "dice_mean",
            "iou_mean",
            "precision_mean",
            "recall_mean",
            "boundary_iou_mean",
            "zero_dice_images",
        ]
    ]
)
display(
    effects[effects.metric == "dice"][
        [
            "comparison",
            "mean_difference",
            "ci95_low",
            "ci95_high",
            "improved",
            "equal",
            "worsened",
            "permutation_p_holm",
        ]
    ]
)
print(json.dumps(selection, indent=2, ensure_ascii=False))


## 7. Ver os resultados

O primeiro gráfico resume desempenho e efeitos pareados. O primeiro painel mantém os 12 casos congelados e todos os modos; o segundo destaca apenas o modo selecionado. Verde é acerto, laranja é falso positivo e vermelho é falso negativo.


In [ ]:
from PIL import Image

display(Image.open(RUN / "sam_validation_summary.png"))
display(Image.open(RUN / "fixed_validation_panel_all_modes.png"))
display(Image.open(RUN / "fixed_validation_panel_selected.png"))


## 8. Empacotar a execução

O ZIP contém métricas, auditoria, gráficos e painéis. Checkpoints e Dataset não são copiados.


In [ ]:
export_dir = PROJECT_ROOT / "exports/day13"
export_dir.mkdir(parents=True, exist_ok=True)
archive = shutil.make_archive(str(export_dir / f"sam_validation_{RUN.name}"), "zip", RUN)
print("Arquivo exportado:", archive)


## Interpretação

A modalidade de SAM é escolhida na validação, e seu efeito final é estimado no teste interno. A modalidade selecionada é fixada antes da avaliação final.
